# Decoder controls: DeepLabV3+ and SegFormer with the recipe of the primary U-Net

Tests whether the advantage of the dense U-Net on video comes from dense per-pixel segmentation in general or from the
U-Net decoder. Both networks are trained exactly like the primary U-Net (156 hoof windows without the frame of the
internal test recording, three classes, windows from claw boxes enlarged by 1.0-1.6, full augmentation, 40 epochs,
checkpoint selection on the six validation photographs, seeds 0-2). Only the network differs:

* **DeepLabV3+**, same ImageNet ConvNeXt-Tiny encoder as the U-Net (output stride 16, ASPP, decoder at 1/4 resolution)
* **SegFormer-B2**, Mix Transformer encoder (ImageNet), all-MLP decoder at 1/4 resolution

Runtime > Change runtime type > GPU (A100 or L4, which support bfloat16, as for the earlier notebooks) before starting. Each run takes about 10-15 min. Finished runs are
skipped, so after a disconnect simply run all cells again.

In [ ]:
# 1. GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Google Drive (results survive a disconnect)
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/angusclaw_arch'
os.makedirs(f'{WORK}/runs', exist_ok=True)
print(WORK)

In [ ]:
# 3. Software: segmentation_models_pytorch as in the paper (torch and timm of Colab are used)
!pip -q install segmentation-models-pytorch==0.6.0
import torch, timm, segmentation_models_pytorch as smp
print('torch', torch.__version__, 'timm', timm.__version__, 'smp', smp.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'no GPU: Runtime > Change runtime type'

In [ ]:
# 4. Package: put arch_package.zip into the Drive folder above, or upload it when asked
import shutil, zipfile
PKG = f'{WORK}/arch_package.zip'
if not os.path.exists(PKG):
    from google.colab import files
    uploaded = files.upload()          # choose arch_package.zip
    shutil.move(next(iter(uploaded)), PKG)
shutil.rmtree('/content/pkg', ignore_errors=True)
with zipfile.ZipFile(PKG) as z:
    z.extractall('/content/pkg')
for s in ('train', 'valid', 'test'):
    print(s, len(os.listdir(f'/content/pkg/data/ClawSet/{s}/images')), 'images (expected 124, 15, 11)')

In [ ]:
# 5. Training function: primary-U-Net flags, only --arch and --encoder differ
import subprocess
PRIMARY = ['--size', '320', '--epochs', '40', '--val_photos_only',
           '--exclude_train_source', '20261007_134710-003', '--box_scale', '1.6', '--workers', '4']

def train(arch, encoder, name, seeds=(0, 1, 2)):
    for seed in seeds:
        run = f'{name}_s{seed}'
        if os.path.exists(f'{WORK}/runs/{run}_metrics.json'):
            print(run, 'done, skipped')
            continue
        cmd = ['python', '-W', 'ignore', 'scripts/train_variants.py', '--arch', arch, '--encoder', encoder,
               *PRIMARY, '--seed', str(seed), '--name', run, '--out', f'{WORK}/runs']
        print(' '.join(cmd), flush=True)
        with open(f'{WORK}/runs/{run}.out', 'w') as log:
            r = subprocess.run(cmd, cwd='/content/pkg', stdout=log, stderr=subprocess.STDOUT,
                               env={**os.environ, 'CLAWSEG_DATA': '/content/pkg/data/ClawSet'})
        print(run, 'exit', r.returncode, '|', open(f'{WORK}/runs/{run}.out').read().strip().splitlines()[-1], flush=True)

In [ ]:
# 6. DeepLabV3+ with the ConvNeXt-Tiny encoder of the U-Net, seeds 0-2 (about 30-45 min)
train('deeplabv3plus', 'convnext_tiny', 'dlv3p_cnx')

In [ ]:
# 7. SegFormer-B2, seeds 0-2 (about 30-45 min)
train('segformer', 'mit_b2', 'segformer_b2')

In [ ]:
# 8. Check: best epoch and validation IoU of each run (the primary U-Net reached claw IoU of about 0.69)
import json, glob
for p in sorted(glob.glob(f'{WORK}/runs/*_metrics.json')):
    m = json.load(open(p))
    print(f"{os.path.basename(p)[:-13]:16s} epoch {m['best_epoch']:2d}  native {m['valid']['native']}  "
          f"video {m['valid']['video']}  {m['params_m']} M  {m['train_seconds']} s")

In [ ]:
# 9. Package the trained models and logs (no images) and download them
zip_path = f'{WORK}/arch_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(glob.glob(f'{WORK}/runs/*')):
        if p.endswith(('.pt', '_metrics.json', '_log.csv', '_train_config.json', '.out')):
            z.write(p, os.path.basename(p))
    z.writestr('versions.txt', f'torch {torch.__version__}\ntimm {timm.__version__}\nsmp {smp.__version__}\n'
               + subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                                capture_output=True, text=True).stdout)
print(zip_path, round(os.path.getsize(zip_path) / 1e6), 'MB')
from google.colab import files
files.download(zip_path)